# HinglishID-Router: Module M1.7 — Interactive Tagger Demonstration

**Module:** M1.7 (Week-1 Review & Tagger Demo)  
**Models Evaluated:** Baseline Lookup ($M_0$), MultinomialNB ($M_1$), and Logistic Regression ($M_1$)  
**Downstream Application:** E-Commerce Romanized Hindi-English Customer Support Routing  

### Demonstration Highlights:
- **Entity & Noise Masking:** Regex masking of `<URL>`, `<EMOJI>`, and `<ORDER_ID>`
- **Character Elongation Reduction:** `plzzzzz` $\to$ `plzz`
- **Word-Level Language Tagging:** Classifies each word token as `HI`, `EN`, or `OTHER`
- **Code-Mixing Index (CMI):** Calculates Gambäck & Das (2014) CMI score ($0.0 = \text{monolingual}$, $50.0 = \text{balanced bilingual}$)
- **Rich Visual Badges:** HTML token visualization displaying language tags and class probabilities.

In [1]:
import sys
from pathlib import Path
from IPython.display import HTML, display
import pandas as pd

# Ensure repository root is on sys.path
sys.path.append(str(Path('..').resolve()))

from src.demo_tagger import HinglishDemoTagger, SAMPLE_QUERIES
print('HinglishDemoTagger imported successfully!')

HinglishDemoTagger imported successfully!


## 1. Load Trained Production Model
We load the tuned **LogisticRegression (char_wb 1-4)** tagger (`models/lid_lr.joblib`), which achieved **97.34% Dev Accuracy** and **77.87% OOV Accuracy** in Module M1.6.

In [2]:
tagger_demo = HinglishDemoTagger(model_path='../models/lid_lr.joblib')
print('Model loaded and preprocessor configured.')

Model loaded and preprocessor configured.


## 2. Interactive Demonstration Across 6 Customer Support Intents
We evaluate 10 realistic customer support queries covering all 6 pre-registered intent categories:
1. `refund`
2. `cancel_order`
3. `order_not_delivered`
4. `delivery_delay`
5. `payment_failed`
6. `other`

In [3]:
results_summary = []

for q in SAMPLE_QUERIES:
    res = tagger_demo.tag_query(q['text'])
    html_pills = tagger_demo.format_html_pills(res)
    
    display(HTML(f'''
    <div style="border: 1px solid #e1e4e8; border-radius: 8px; padding: 14px; margin-bottom: 16px; background-color: #fafbfc;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
            <span style="font-size: 14px; font-weight: bold; color: #0366d6;">Query #{q['id']} — Intent: <span style="color: #24292e; text-transform: uppercase;">{q['intent']}</span></span>
            <span style="font-size: 12px; background: #e1e4e8; padding: 2px 8px; border-radius: 12px;">CMI: <b>{res['cmi']:.1f}</b> | Code-Mixed: <b>{res['is_code_mixed']}</b></span>
        </div>
        <div style="font-size: 13px; color: #586069; margin-bottom: 8px;"><b>Raw:</b> <i>{q['text']}</i></div>
        <div style="margin-top: 6px;">{html_pills}</div>
    </div>
    '''))
    
    results_summary.append({
        'Query ID': q['id'],
        'Intent': q['intent'],
        'Token Count': len(res['tokens']),
        'Hindi (HI)': res['language_distribution']['HI'],
        'English (EN)': res['language_distribution']['EN'],
        'Other': res['language_distribution']['OTHER'],
        'CMI Score': res['cmi'],
        'Code-Mixed': res['is_code_mixed'],
    })

## 3. Query Statistics & Code-Mixing Summary Table

In [4]:
df_summary = pd.DataFrame(results_summary)
display(df_summary)

,Query ID,Intent,Token Count,Hindi (HI),English (EN),Other,CMI Score,Code-Mixed
0,1,refund,14,8,3,3,27.27,True
1,2,refund,11,2,7,2,22.22,True
2,3,cancel_order,12,6,4,2,40.00,True
3,4,cancel_order,12,0,9,3,0.00,False
4,5,delivery_delay,13,7,4,2,36.36,True
5,6,delivery_delay,14,3,9,2,25.00,True
6,7,order_not_delivered,11,6,4,1,40.00,True
7,8,payment_failed,13,4,8,1,33.33,True
8,9,payment_failed,10,2,6,2,25.00,True
9,10,other,15,3,8,4,27.27,True


## 4. Frozen Test Set Integrity Verification (Day 7 Protocol)
In compliance with project specifications, we verify that the test partition (`data/processed/test.conll`) has remained strictly **FROZEN** and has not been accessed during model selection or hyperparameter tuning.

In [5]:
import hashlib

test_path = Path('../data/processed/test.conll')
assert test_path.is_file(), 'test.conll must exist!'

sha256_hash = hashlib.sha256()
with open(test_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b''):
        sha256_hash.update(byte_block)

line_count = sum(1 for _ in open(test_path, encoding='utf-8'))
print(f'Test Partition: {test_path}')
print(f'Total Lines:    {line_count:,}')
print(f'SHA256 Hash:    {sha256_hash.hexdigest()}')
print('STATUS: [VERIFIED] test.conll is intact and remains completely FROZEN until Day 10.')

Test Partition: ..\data\processed\test.conll
Total Lines:    140,014
SHA256 Hash:    a3ce094d776066cafebc9cda2d00a42521b7b838d66af075296af2d3843d8923
STATUS: [VERIFIED] test.conll is intact and remains completely FROZEN until Day 10.
